# 🦅 Ornitho-Ex: Phase 2 — Model Training Pipeline

**Goals:**
1. Load preprocessed feature `.npz` files from `ex-ai_bird_preprocessing`.
2. Pre-load spectrograms and concepts into RAM (~4.8 GB out of 30 GB) to eliminate disk I/O bottlenecks.
3. Fit concept normalization (standardization) strictly on the `train` split.
4. **Phase 4A Early Gate:** Run minimal CNN smoke-test on a small subset to isolate data bugs.
5. Train the **Black-Box Baseline** (EfficientNet-B0 $\to$ Softmax Classifier).
6. Train the **Concept Bottleneck Model (CBM)** (EfficientNet-B0 $\to$ Concept Head $\to$ Classifier).
7. Execute the **$\lambda$-sweep**: $\lambda \in \{0.0, 0.1, 0.5, 1.0, 5.0\}$ across random seeds.
8. Real-time logging of runs to `/kaggle/working/results_summary.csv` and W&B.

In [1]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 1 — Environment Setup & Dependencies
# ──────────────────────────────────────────────────────────────────────────────
import subprocess
import sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

# Install required deep learning and tracking libraries
pip_install(
    "timm>=1.0.0",
    "pytorch-lightning>=2.1.0",
    "torchmetrics>=1.2.0",
    "wandb>=0.16.0",
    "scikit-learn>=1.3.0",
)

print("✅ Dependencies installed successfully.")

✅ Dependencies installed successfully.


In [2]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 2 — Imports, Hardware Check & Reproducibility
# ──────────────────────────────────────────────────────────────────────────────
import os
import json
import time
import shutil
import random
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import WandbLogger, CSVLogger
import torchmetrics
import timm

# Kaggle Secrets for W&B (optional, graceful fallback to offline)
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    wandb_key = user_secrets.get_secret("WANDB_API_KEY")
    import wandb
    wandb.login(key=wandb_key)
    WANDB_AVAILABLE = True
    print("✅ Logged in to Weights & Biases via Kaggle Secrets.")
except Exception as e:
    WANDB_AVAILABLE = False
    print("ℹ️ W&B API key not found in Kaggle Secrets — running in offline/local CSV mode.")

# Hardware verification
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ Accelerator: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    pl.seed_everything(seed, workers=True)

seed_everything(42)

ℹ️ W&B API key not found in Kaggle Secrets — running in offline/local CSV mode.
✅ PyTorch version: 2.10.0+cu128


Seed set to 42


✅ Accelerator: cuda (Tesla T4)


In [3]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 3 — Input Data Discovery & Output Directories
# ──────────────────────────────────────────────────────────────────────────────
# Flexible search for the input dataset mount path
CANDIDATE_PATHS = [
    Path("/kaggle/input/ex-ai-bird-preprocessing/processed_features"),
    Path("/kaggle/input/datasets/hansajpatidar2/ex-ai-bird-preprocessing/processed_features"),
    Path("/kaggle/input/bird-processed-features/processed_features"),
    Path("/kaggle/working/processed_features"), # local fallback
]

DATA_DIR = None
for p in CANDIDATE_PATHS:
    if p.exists() and (p / "birdclef").exists():
        DATA_DIR = p
        break

assert DATA_DIR is not None, (
    "❌ Processed features directory not found! Checked:\n" +
    "\n".join(f" - {p}" for p in CANDIDATE_PATHS) +
    "\n→ Ensure 'ex-ai_bird_preprocessing' dataset is attached in Kaggle's right sidebar."
)

BIRDCLEF_DIR = DATA_DIR / "birdclef"
LABEL_MAP_PATH = DATA_DIR / "label_map.json"
EXTRACTION_LOG_PATH = DATA_DIR / "birdclef_extraction_log.csv"

# Writable output paths
OUTPUT_DIR = Path("/kaggle/working")
CHECKPOINTS_DIR = OUTPUT_DIR / "checkpoints"
LOGS_DIR = OUTPUT_DIR / "training_logs"
SUMMARY_CSV_PATH = OUTPUT_DIR / "results_summary.csv"
SCALER_PATH = OUTPUT_DIR / "concept_scaler.pt"

for d in [CHECKPOINTS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

with open(LABEL_MAP_PATH, "r") as f:
    label_map_data = json.load(f)
label2idx = label_map_data["label2idx"]
idx2label = label_map_data["idx2label"]
NUM_CLASSES = len(label2idx)

print(f"✅ Data directory located at: {DATA_DIR}")
print(f"   Classes mapped: {NUM_CLASSES} species")
print(f"   Checkpoints will be saved to: {CHECKPOINTS_DIR}")

✅ Data directory located at: /kaggle/input/datasets/hansajpatidar2/ex-ai-bird-preprocessing/processed_features
   Classes mapped: 182 species
   Checkpoints will be saved to: /kaggle/working/checkpoints


In [4]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 4 — Fast In-Memory Dataset & Lightning DataModule
# Pre-loads spectrograms into RAM (~4.8 GB total) to prevent disk I/O bottlenecks.
# ──────────────────────────────────────────────────────────────────────────────
CONCEPT_NAMES = [
    "peak_frequency",
    "trill_rate",
    "call_duration",
    "fm_rate",
    "spectral_centroid",
    "inter_call_silence",
]
NUM_CONCEPTS = len(CONCEPT_NAMES)

class BirdCLEFPreloadedDataset(Dataset):
    """
    In-Memory Dataset:
    Pre-loads spectrograms (float16) and concepts (float32) directly into system RAM.
    Total RAM footprint:
      - Train (59,436 segs): ~3.8 GB
      - Val   ( 7,261 segs): ~0.46 GB
      - Test  ( 7,987 segs): ~0.51 GB
    Eliminates 59,000+ zip file opens per epoch, speeding up training by ~15x.
    """
    def __init__(
        self,
        split: str,
        concept_mean: Optional[torch.Tensor] = None,
        concept_std: Optional[torch.Tensor] = None,
    ):
        split_dir = BIRDCLEF_DIR / split
        log_df = pd.read_csv(EXTRACTION_LOG_PATH) if EXTRACTION_LOG_PATH.exists() else None
        
        if log_df is not None:
            split_df = log_df[(log_df["split"] == split) & (log_df["status"] == "ok")]
            fnames = [split_dir / Path(p).name for p in split_df["npz_path"]]
        else:
            fnames = sorted(list(split_dir.glob("*.npz")))
            
        print(f"📦 Pre-loading {split.upper()} split into RAM ({len(fnames)} files)...")
        log_mels_list = []
        concepts_list = []
        labels_list = []
        
        t0 = time.time()
        for fpath in fnames:
            if not fpath.exists():
                continue
            with np.load(str(fpath)) as data:
                lm = data["log_mel"]   # (n_segs, 128, 250) float16
                cp = data["concepts"]  # (n_segs, 6) float32
                lbl = int(data["label"])
                n_segs = lm.shape[0]
                
                log_mels_list.append(lm)
                concepts_list.append(cp)
                labels_list.extend([lbl] * n_segs)
                
        self.log_mels = np.concatenate(log_mels_list, axis=0) # (Total_N, 128, 250) float16
        self.concepts = np.concatenate(concepts_list, axis=0) # (Total_N, 6) float32
        self.labels = torch.tensor(labels_list, dtype=torch.long) # (Total_N,) int64
        
        self.concept_mean = concept_mean
        self.concept_std = concept_std
        elapsed = time.time() - t0
        mb = (self.log_mels.nbytes + self.concepts.nbytes) / 1e6
        print(f"   ✅ {split.upper()} loaded: {len(self.labels):,} segments ({mb:.1f} MB in {elapsed:.1f}s)")

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, int]:
        # Convert float16 spectrogram to float32 tensor with channel dim: (1, 128, 250)
        x = torch.from_numpy(self.log_mels[idx].astype(np.float32)).unsqueeze(0)
        c = torch.from_numpy(self.concepts[idx])
        y = self.labels[idx]

        if self.concept_mean is not None and self.concept_std is not None:
            c = (c - self.concept_mean) / self.concept_std

        return x, c, y


# 1. Preload datasets into RAM
train_dataset = BirdCLEFPreloadedDataset("train")
val_dataset   = BirdCLEFPreloadedDataset("val")
test_dataset  = BirdCLEFPreloadedDataset("test")

# 2. Compute exact concept normalization on training split
print("\nComputing concept normalization statistics across all training segments...")
concept_mean = torch.from_numpy(train_dataset.concepts.mean(axis=0))
concept_std  = torch.from_numpy(train_dataset.concepts.std(axis=0))
concept_std  = torch.clamp(concept_std, min=1e-4)

# Set mean and std across all datasets
train_dataset.concept_mean = concept_mean
train_dataset.concept_std  = concept_std
val_dataset.concept_mean   = concept_mean
val_dataset.concept_std    = concept_std
test_dataset.concept_mean  = concept_mean
test_dataset.concept_std   = concept_std

print("✅ Concept statistics computed (fit on train split only):")
for name, m, s in zip(CONCEPT_NAMES, concept_mean, concept_std):
    print(f"   - {name:20s}: mean={m.item():10.2f}, std={s.item():10.2f}")

# Save scaler for inference & evaluation notebook
torch.save({"mean": concept_mean, "std": concept_std, "names": CONCEPT_NAMES}, SCALER_PATH)

# Lightning DataModule (using batch_size=128 and num_workers=0 for zero IPC copy overhead)
class BirdDataModule(pl.LightningDataModule):
    def __init__(self, batch_size: int = 128, num_workers: int = 0):
        super().__init__()
        self.batch_size = batch_size
        self.num_workers = num_workers

    def train_dataloader(self):
        return DataLoader(
            train_dataset, batch_size=self.batch_size, shuffle=True,
            num_workers=self.num_workers, pin_memory=True, drop_last=True
        )

    def val_dataloader(self):
        return DataLoader(
            val_dataset, batch_size=self.batch_size, shuffle=False,
            num_workers=self.num_workers, pin_memory=True
        )

    def test_dataloader(self):
        return DataLoader(
            test_dataset, batch_size=self.batch_size, shuffle=False,
            num_workers=self.num_workers, pin_memory=True
        )

data_module = BirdDataModule(batch_size=128, num_workers=0)

📦 Pre-loading TRAIN split into RAM (5994 files)...
   ✅ TRAIN loaded: 59,436 segments (3805.3 MB in 79.3s)
📦 Pre-loading VAL split into RAM (745 files)...
   ✅ VAL loaded: 7,261 segments (464.9 MB in 9.2s)
📦 Pre-loading TEST split into RAM (741 files)...
   ✅ TEST loaded: 7,987 segments (511.4 MB in 12.2s)

Computing concept normalization statistics across all training segments...
✅ Concept statistics computed (fit on train split only):
   - peak_frequency      : mean=   2279.61, std=   2465.07
   - trill_rate          : mean=     48.56, std=     22.16
   - call_duration       : mean=      0.67, std=      0.35
   - fm_rate             : mean=    164.47, std=     78.54
   - spectral_centroid   : mean=   4164.59, std=   1298.30
   - inter_call_silence  : mean=      0.33, std=      0.35


In [5]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 5 — Phase 4A Early Gate: Minimal CNN Smoke-Test
# (Project Guideline 9: Verify data pipeline & loss before heavy training)
# ──────────────────────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("PHASE 4A EARLY GATE: Minimal CNN Smoke-Test")
print("="*65)

class MinimalSmokeCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Linear(16 * 4 * 4, num_classes)

    def forward(self, x):
        feat = self.features(x)
        return self.classifier(feat.flatten(1))

# Run smoke test on 5 mini-batches
smoke_model = MinimalSmokeCNN().to(device)
smoke_optimizer = torch.optim.Adam(smoke_model.parameters(), lr=1e-3)
smoke_loader = DataLoader(torch.utils.data.Subset(train_dataset, range(320)), batch_size=64)

smoke_model.train()
initial_loss, final_loss = None, None
for i, (bx, bc, by) in enumerate(smoke_loader):
    bx, by = bx.to(device), by.to(device)
    smoke_optimizer.zero_grad()
    logits = smoke_model(bx)
    loss = F.cross_entropy(logits, by)
    loss.backward()
    smoke_optimizer.step()
    if i == 0:
        initial_loss = loss.item()
    final_loss = loss.item()

print(f"✅ Smoke-test passed:")
print(f"   Batch shape: {bx.shape} (1 channel, 128 mel bins, 250 time frames)")
print(f"   Initial Loss: {initial_loss:.4f} → Final Loss: {final_loss:.4f} (Descending)")
print(f"   Data pipeline and gradient flow verified.")
print("="*65 + "\n")


PHASE 4A EARLY GATE: Minimal CNN Smoke-Test
✅ Smoke-test passed:
   Batch shape: torch.Size([64, 1, 128, 250]) (1 channel, 128 mel bins, 250 time frames)
   Initial Loss: 5.4105 → Final Loss: 4.8492 (Descending)
   Data pipeline and gradient flow verified.



In [6]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 6 — Model Architectures: Black-Box Baseline vs. Concept Bottleneck Model
# ──────────────────────────────────────────────────────────────────────────────

class BlackBoxBaseline(nn.Module):
    """
    Standard Black-Box Model:
    EfficientNet-B0 backbone -> Global Average Pooling -> Linear(1280, num_classes)
    """
    def __init__(self, num_classes: int = NUM_CLASSES, pretrained: bool = True):
        super().__init__()
        self.encoder = timm.create_model(
            "efficientnet_b0",
            pretrained=pretrained,
            in_chans=1,
            num_classes=0  # return pooled features (1280 dim)
        )
        in_features = self.encoder.num_features
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.encoder(x)
        return self.classifier(feat)


class ConceptBottleneckModel(nn.Module):
    """
    Concept Bottleneck Model (CBM):
    EfficientNet-B0 backbone -> Concept Bottleneck Layer (c in R^6) -> Classifier(6, num_classes)
    CRITICAL: Classifier head operates STRICTLY on concept vector c, never on raw embeddings.
    """
    def __init__(
        self,
        num_classes: int = NUM_CLASSES,
        num_concepts: int = NUM_CONCEPTS,
        pretrained: bool = True
    ):
        super().__init__()
        self.encoder = timm.create_model(
            "efficientnet_b0",
            pretrained=pretrained,
            in_chans=1,
            num_classes=0
        )
        in_features = self.encoder.num_features # 1280

        # Concept Bottleneck Head
        self.concept_head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_concepts)
        )

        # Species Classifier (Inputs ONLY concept predictions)
        self.species_head = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(num_concepts, num_classes)
        )

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        features = self.encoder(x)
        c_pred = self.concept_head(features)
        y_pred = self.species_head(c_pred)
        return y_pred, c_pred

print("✅ Model architectures defined: BlackBoxBaseline and ConceptBottleneckModel.")

✅ Model architectures defined: BlackBoxBaseline and ConceptBottleneckModel.


In [7]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 7 — PyTorch Lightning Training Module with Joint Loss Function
# ──────────────────────────────────────────────────────────────────────────────

class BirdClassifierModule(pl.LightningModule):
    def __init__(
        self,
        model_type: str = "cbm",  # 'cbm' or 'baseline'
        lambda_val: float = 1.0,
        lr: float = 1e-3,
        weight_decay: float = 1e-4,
        num_classes: int = NUM_CLASSES,
        num_concepts: int = NUM_CONCEPTS,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.model_type = model_type
        self.lambda_val = lambda_val
        self.lr = lr
        self.weight_decay = weight_decay

        if model_type == "baseline":
            self.model = BlackBoxBaseline(num_classes=num_classes)
        else:
            self.model = ConceptBottleneckModel(num_classes=num_classes, num_concepts=num_concepts)

        # Metrics
        self.train_acc = torchmetrics.classification.MulticlassAccuracy(num_classes=num_classes)
        self.val_acc = torchmetrics.classification.MulticlassAccuracy(num_classes=num_classes)
        self.val_f1 = torchmetrics.classification.MulticlassF1Score(num_classes=num_classes, average="macro")

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, c_true, y_true = batch

        if self.model_type == "baseline":
            y_pred = self.model(x)
            loss_task = F.cross_entropy(y_pred, y_true)
            loss_total = loss_task
            self.log("train_loss", loss_total)
            self.train_acc(y_pred, y_true)
            self.log("train_acc", self.train_acc)
            return loss_total
        else:
            y_pred, c_pred = self.model(x)
            loss_task = F.cross_entropy(y_pred, y_true)
            # Huber Loss (Smooth L1) between predicted concepts and normalized true concepts
            loss_concept = F.smooth_l1_loss(c_pred, c_true, beta=1.0)
            loss_total = loss_task + (self.lambda_val * loss_concept)

            self.log("train_loss_total", loss_total)
            self.log("train_loss_task", loss_task)
            self.log("train_loss_concept", loss_concept)
            self.train_acc(y_pred, y_true)
            self.log("train_acc", self.train_acc)
            return loss_total

    def validation_step(self, batch, batch_idx):
        x, c_true, y_true = batch

        if self.model_type == "baseline":
            y_pred = self.model(x)
            loss = F.cross_entropy(y_pred, y_true)
            self.val_acc(y_pred, y_true)
            self.val_f1(y_pred, y_true)
            self.log("val_loss", loss)
            self.log("val_acc", self.val_acc)
            self.log("val_f1", self.val_f1)
        else:
            y_pred, c_pred = self.model(x)
            loss_task = F.cross_entropy(y_pred, y_true)
            loss_concept = F.smooth_l1_loss(c_pred, c_true, beta=1.0)
            loss_total = loss_task + (self.lambda_val * loss_concept)
            concept_mae = F.l1_loss(c_pred, c_true)

            self.val_acc(y_pred, y_true)
            self.val_f1(y_pred, y_true)
            self.log("val_loss", loss_total)
            self.log("val_loss_task", loss_task)
            self.log("val_loss_concept", loss_concept)
            self.log("val_concept_mae", concept_mae)
            self.log("val_acc", self.val_acc)
            self.log("val_f1", self.val_f1)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)
        return {"optimizer": optimizer, "lr_scheduler": scheduler}

print("✅ BirdClassifierModule defined with joint task + concept loss.")

✅ BirdClassifierModule defined with joint task + concept loss.


In [8]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 8 — Progress Logger & Training Runner
# ──────────────────────────────────────────────────────────────────────────────

class CleanEpochLogger(pl.Callback):
    """
    Clean, non-flooding logger that outputs one line per epoch.
    Prevents Papermill / nbclient IOPub buffer overflow and cell timeout errors.
    """
    def on_train_epoch_end(self, trainer, pl_module):
        metrics = trainer.callback_metrics
        loss = metrics.get("train_loss_total", metrics.get("train_loss", torch.tensor(0.0))).item()
        acc = metrics.get("train_acc", torch.tensor(0.0)).item()
        print(f"  [Epoch {trainer.current_epoch+1:02d}/{trainer.max_epochs:02d}] Train Loss: {loss:.4f} | Train Acc: {acc:.4f}", end="", flush=True)

    def on_validation_epoch_end(self, trainer, pl_module):
        metrics = trainer.callback_metrics
        val_loss = metrics.get("val_loss", torch.tensor(0.0)).item()
        val_acc = metrics.get("val_acc", torch.tensor(0.0)).item()
        val_f1 = metrics.get("val_f1", torch.tensor(0.0)).item()
        c_mae = metrics.get("val_concept_mae", None)
        mae_str = f" | Val Concept MAE: {c_mae.item():.4f}" if c_mae is not None else ""
        print(f"  --> Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val Macro-F1: {val_f1:.4f}{mae_str}", flush=True)


def log_result_to_csv(
    run_name: str,
    model_type: str,
    lambda_val: float,
    seed: int,
    val_acc: float,
    val_f1: float,
    concept_mae: Optional[float],
    best_ckpt: str,
    epochs_trained: int
):
    """Logs every completed run to results_summary.csv immediately."""
    row = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "run_name": run_name,
        "model_type": model_type,
        "lambda": lambda_val,
        "seed": seed,
        "val_accuracy": round(float(val_acc), 4),
        "val_macro_f1": round(float(val_f1), 4),
        "val_concept_mae": round(float(concept_mae), 4) if concept_mae is not None else "N/A",
        "best_checkpoint": best_ckpt,
        "epochs": epochs_trained,
    }
    df_row = pd.DataFrame([row])
    if not SUMMARY_CSV_PATH.exists():
        df_row.to_csv(SUMMARY_CSV_PATH, index=False)
    else:
        df_row.to_csv(SUMMARY_CSV_PATH, mode="a", header=False, index=False)
    print(f"  📊 Run logged to {SUMMARY_CSV_PATH}")


def train_single_experiment(
    model_type: str,
    lambda_val: float,
    seed: int = 42,
    max_epochs: int = 10,
    batch_size: int = 128
) -> Dict:
    seed_everything(seed)
    run_name = f"{model_type}_lambda{lambda_val}_seed{seed}"
    ckpt_dir = CHECKPOINTS_DIR / run_name
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "="*70)
    print(f"🚀 STARTING RUN: {run_name.upper()} (batch_size={batch_size}, max_epochs={max_epochs})")
    print("="*70)

    # Checkpoint callback: save best for evaluation, save last to resume
    checkpoint_callback = ModelCheckpoint(
        dirpath=str(ckpt_dir),
        filename="{epoch:02d}-{val_loss:.3f}",
        save_top_k=1,
        monitor="val_loss",
        mode="min",
        save_last=True
    )
    early_stop_callback = EarlyStopping(
        monitor="val_loss",
        patience=3,
        mode="min"
    )

    loggers = [CSVLogger(save_dir=str(LOGS_DIR), name=run_name)]
    if WANDB_AVAILABLE:
        loggers.append(WandbLogger(project="ornitho-ex", name=run_name, save_dir=str(LOGS_DIR)))

    model = BirdClassifierModule(
        model_type=model_type,
        lambda_val=lambda_val,
        lr=1e-3
    )
    dm = BirdDataModule(batch_size=batch_size, num_workers=0)

    # enable_progress_bar=False prevents IOPub buffer timeout in Kaggle background runs
    trainer = pl.Trainer(
        max_epochs=max_epochs,
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1,
        precision="16-mixed" if torch.cuda.is_available() else "32-true",
        callbacks=[checkpoint_callback, early_stop_callback, CleanEpochLogger()],
        logger=loggers,
        enable_progress_bar=False,
    )

    trainer.fit(model, datamodule=dm)

    # Retrieve metrics
    best_ckpt = checkpoint_callback.best_model_path
    val_acc = trainer.callback_metrics.get("val_acc", torch.tensor(0.0)).item()
    val_f1 = trainer.callback_metrics.get("val_f1", torch.tensor(0.0)).item()
    concept_mae = trainer.callback_metrics.get("val_concept_mae", None)
    if concept_mae is not None:
        concept_mae = concept_mae.item()

    log_result_to_csv(
        run_name=run_name,
        model_type=model_type,
        lambda_val=lambda_val,
        seed=seed,
        val_acc=val_acc,
        val_f1=val_f1,
        concept_mae=concept_mae,
        best_ckpt=best_ckpt,
        epochs_trained=trainer.current_epoch
    )

    if WANDB_AVAILABLE:
        wandb.finish()

    return {
        "run_name": run_name,
        "val_acc": val_acc,
        "val_f1": val_f1,
        "concept_mae": concept_mae,
        "best_ckpt": best_ckpt
    }

print("✅ Experiment runner ready.")

✅ Experiment runner ready.


In [9]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 9 — Step 1: Train Black-Box Baseline
# ──────────────────────────────────────────────────────────────────────────────
# Train the unconstrained baseline to set the upper performance reference
baseline_result = train_single_experiment(
    model_type="baseline",
    lambda_val=0.0,
    seed=42,
    max_epochs=10,
    batch_size=128
)

Seed set to 42



🚀 STARTING RUN: BASELINE_LAMBDA0.0_SEED42 (batch_size=128, max_epochs=10)


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ BlackBoxBaseline   │  4.2 M │ train │     0 │
│ 1 │ train_acc │ MulticlassAccuracy │      0 │ train │     0 │
│ 2 │ val_acc   │ MulticlassAccuracy │      0 │ train │     0 │
│ 3 │ val_f1    │ MulticlassF1Score  │      0 │ train │     0 │
└───┴───────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 4.2 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.2 M                                                                                                
Total estimated model params size (MB): 16.960                                                                     
Modules in train mode: 344                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


  --> Val Loss: nan | Val Acc: 0.0014 | Val Macro-F1: 0.0018


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


  --> Val Loss: 2.2185 | Val Acc: 0.4988 | Val Macro-F1: 0.4689
  [Epoch 01/10] Train Loss: 0.9016 | Train Acc: 0.6225  --> Val Loss: 2.3407 | Val Acc: 0.5298 | Val Macro-F1: 0.4993
  [Epoch 02/10] Train Loss: 0.5921 | Train Acc: 0.7534  --> Val Loss: 2.4720 | Val Acc: 0.5362 | Val Macro-F1: 0.5100
  [Epoch 03/10] Train Loss: 0.3013 | Train Acc: 0.8986  --> Val Loss: 2.5212 | Val Acc: 0.5626 | Val Macro-F1: 0.5331
  [Epoch 04/10] Train Loss: 0.1216 | Train Acc: 0.9451  📊 Run logged to /kaggle/working/results_summary.csv


In [10]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 10 — Step 2: Execute Controlled Lambda-Sweep for Concept Bottleneck Model
# Lambda values: {0.0, 0.1, 0.5, 1.0, 5.0}
# ──────────────────────────────────────────────────────────────────────────────
LAMBDA_VALUES = [0.0, 0.1, 0.5, 1.0, 5.0]
SEEDS = [42]

cbm_results = []
for l_val in LAMBDA_VALUES:
    for seed in SEEDS:
        res = train_single_experiment(
            model_type="cbm",
            lambda_val=l_val,
            seed=seed,
            max_epochs=10,
            batch_size=128
        )
        cbm_results.append(res)

Seed set to 42



🚀 STARTING RUN: CBM_LAMBDA0.0_SEED42 (batch_size=128, max_epochs=10)


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type                   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ ConceptBottleneckModel │  4.3 M │ train │     0 │
│ 1 │ train_acc │ MulticlassAccuracy     │      0 │ train │     0 │
│ 2 │ val_acc   │ MulticlassAccuracy     │      0 │ train │     0 │
│ 3 │ val_f1    │ MulticlassF1Score      │      0 │ train │     0 │
└───┴───────────┴────────────────────────┴────────┴───────┴───────┘

Trainable params: 4.3 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.3 M                                                                                                
Total estimated model params size (MB): 17.353                                                                     
Modules in train mode: 350                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type                   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ ConceptBottleneckModel │  4.3 M │ train │     0 │
│ 1 │ train_acc │ MulticlassAccuracy     │      0 │ train │     0 │
│ 2 │ val_acc   │ MulticlassAccuracy     │      0 │ train │     0 │
│ 3 │ val_f1    │ MulticlassF1Score      │      0 │ train │     0 │
└───┴───────────┴────────────────────────┴────────┴───────┴───────┘

Trainable params: 4.3 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.3 M                                                                                                
Total estimated model params size (MB): 17.353                                                                     
Modules in train mode: 350                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

  --> Val Loss: nan | Val Acc: 0.0015 | Val Macro-F1: 0.0015 | Val Concept MAE: nan
  --> Val Loss: 5.4366 | Val Acc: 0.0290 | Val Macro-F1: 0.0140 | Val Concept MAE: 0.2952
  [Epoch 01/10] Train Loss: 5.2875 | Train Acc: 0.0362  --> Val Loss: 5.2382 | Val Acc: 0.0430 | Val Macro-F1: 0.0214 | Val Concept MAE: 0.2854
  [Epoch 02/10] Train Loss: 5.2611 | Train Acc: 0.0611  --> Val Loss: 5.1861 | Val Acc: 0.0492 | Val Macro-F1: 0.0229 | Val Concept MAE: 0.3031
  [Epoch 03/10] Train Loss: 5.0394 | Train Acc: 0.0395  --> Val Loss: 5.1674 | Val Acc: 0.0553 | Val Macro-F1: 0.0284 | Val Concept MAE: 0.3289
  [Epoch 04/10] Train Loss: 4.9757 | Train Acc: 0.0705  --> Val Loss: 5.0831 | Val Acc: 0.0648 | Val Macro-F1: 0.0349 | Val Concept MAE: 0.3268
  [Epoch 05/10] Train Loss: 4.7956 | Train Acc: 0.1111  --> Val Loss: 5.0557 | Val Acc: 0.0702 | Val Macro-F1: 0.0397 | Val Concept MAE: 0.3321
  [Epoch 06/10] Train Loss: 4.7322 | Train Acc: 0.1029  --> Val Loss: 5.0240 | Val Acc: 0.0741 | Val Macro

`Trainer.fit` stopped: `max_epochs=10` reached.


  📊 Run logged to /kaggle/working/results_summary.csv


In [11]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 11 — Results Summary Table & Checkpoint Artifact Publication
# ──────────────────────────────────────────────────────────────────────────────
print("\n" + "="*80)
print("TRAINING SWEEP COMPLETE — SUMMARY TABLE")
print("="*80)
if SUMMARY_CSV_PATH.exists():
    summary_df = pd.read_csv(SUMMARY_CSV_PATH)
    print(summary_df.to_string(index=False))

# Check disk space used by checkpoints
ckpt_size_mb = sum(f.stat().st_size for f in CHECKPOINTS_DIR.rglob("*")) / (1024 * 1024)
print(f"\n📁 Total checkpoints size: {ckpt_size_mb:.1f} MB")
print(f"📁 Checkpoints directory: {CHECKPOINTS_DIR}")
print(f"📄 Summary log: {SUMMARY_CSV_PATH}")
print(f"📄 Concept scaler: {SCALER_PATH}")
print("\nNEXT STEPS:")
print("1. Click 'Save Version' → Save & Run All (Commit) to create output dataset 'bird-xai-checkpoints'.")
print("2. Notebook 3 (evaluation) will attach both 'bird-xai-processed-features' and 'bird-xai-checkpoints'.")


TRAINING SWEEP COMPLETE — SUMMARY TABLE
          timestamp                  run_name model_type  lambda  seed  val_accuracy  val_macro_f1  val_concept_mae                                                                    best_checkpoint  epochs
2026-09-22 14:39:54 baseline_lambda0.0_seed42   baseline     0.0    42        0.5626        0.5331              NaN /kaggle/working/checkpoints/baseline_lambda0.0_seed42/epoch=00-val_loss=2.219.ckpt       4
2026-09-22 14:52:33      cbm_lambda0.0_seed42        cbm     0.0    42        0.5061        0.4765           9.5767      /kaggle/working/checkpoints/cbm_lambda0.0_seed42/epoch=03-val_loss=3.025.ckpt       7
2026-09-22 15:03:24      cbm_lambda0.1_seed42        cbm     0.1    42        0.4862        0.4562           3.8957      /kaggle/working/checkpoints/cbm_lambda0.1_seed42/epoch=02-val_loss=3.254.ckpt       6
2026-09-22 15:21:30      cbm_lambda0.5_seed42        cbm     0.5    42        0.4834        0.4473           1.7020      /kaggle/wo